# Atividade 3 — Detecção de resíduos recicláveis com YOLO11

Treino de um detector de objetos com a biblioteca **Ultralytics** (modelo base `yolo11n.pt`) no dataset
**Thesis**, do Roboflow Universe: <https://universe.roboflow.com/ed-mikael-bebic-iyvku/thesis-efjq4>
(versão 31, 4 classes: `paper`, `can`, `plastic bottle`, `food-wrapper`; licença CC BY 4.0, autor Ed Mikael Bebic).

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/data-science-unicamp/aplicacoes-data-science/blob/main/atividade3/treino.ipynb)

**Antes de rodar:**
1. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.
2. Na barra lateral, ícone de chave (**Secrets**): crie o secret `ROBOFLOW_API_KEY` com a sua chave do Roboflow
   e ligue o **Acesso ao notebook**. A chave nunca aparece no código.
3. Rode as células em ordem. Tudo é salvo em `MyDrive/atividade3` no Google Drive; se a sessão cair,
   rode de novo as células 1 a 5 e a de treino, que ele continua da última época salva.

## Passo 1 — Ambiente

In [ ]:
# Instala as bibliotecas e confere a versão do Ultralytics e a GPU disponível
!pip install -q ultralytics roboflow

import torch
import ultralytics

ultralytics.checks()
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução > Alterar o tipo > GPU T4"

In [ ]:
# Monta o Google Drive: treino e evidências ficam salvos lá mesmo se a sessão do Colab cair
import os
from google.colab import drive

drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/atividade3"
RUNS = f"{BASE}/runs"            # saída do treino (pesos, gráficos, results.csv)
EVID = f"{BASE}/evidencias"      # tudo que vai para o relatório
os.makedirs(EVID, exist_ok=True)

## Passos 2 e 3 — Dataset do Roboflow Universe

In [ ]:
# Baixa a versão 31 do dataset Thesis no formato YOLOv8 (compatível com YOLO11)
# A chave vem dos Secrets do Colab, nunca escrita no código
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("ed-mikael-bebic-iyvku").project("thesis-efjq4")
dataset = project.version(31).download("yolov8", location="/content/dataset")
ROOT = dataset.location
print("Dataset em:", ROOT)

## Passo 4 — Inspeção do dataset

In [ ]:
# Conteúdo original do data.yaml e contagem de imagens e labels em cada pasta
import glob

linhas = ["data.yaml original (Roboflow):", open(f"{ROOT}/data.yaml").read(), "Estrutura e contagem:"]
for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{ROOT}/{split}/images/*")
    lbls = glob.glob(f"{ROOT}/{split}/labels/*")
    linhas.append(f"{split}/images -> {len(imgs)} imagens | {split}/labels -> {len(lbls)} labels")

texto = "\n".join(linhas)
print(texto)
with open(f"{EVID}/01_estrutura_dataset.txt", "w") as f:
    f.write(texto)

## Passo 5 — Ajuste do data.yaml

In [ ]:
# O data.yaml do Roboflow usa caminhos relativos ("../train/images"), que quebram no Colab.
# Reescreve com caminhos absolutos, mantendo as classes originais.
import yaml

cfg = yaml.safe_load(open(f"{ROOT}/data.yaml"))
NAMES = list(cfg["names"])
DATA = f"{ROOT}/data.yaml"

novo = {
    "train": f"{ROOT}/train/images",
    "val": f"{ROOT}/valid/images",
    "test": f"{ROOT}/test/images",
    "nc": len(NAMES),
    "names": NAMES,
}
with open(DATA, "w") as f:
    yaml.safe_dump(novo, f, sort_keys=False, allow_unicode=True)

for chave in ["train", "val", "test"]:
    assert os.path.isdir(novo[chave]), f"Pasta não encontrada: {novo[chave]}"

print(open(DATA).read())
with open(f"{EVID}/02_data_yaml.txt", "w") as f:
    f.write(open(DATA).read())

## Passo 4 (continuação) — Exemplos anotados

Cada linha de um arquivo `.txt` de label é `classe x_centro y_centro largura altura`, com as coordenadas
normalizadas entre 0 e 1 em relação ao tamanho da imagem. A célula abaixo converte esses valores para pixels
e desenha as caixas em 4 imagens de treino.

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

def le_caixas(label, w, h):
    """Lê um label YOLO e devolve (classe, x1, y1, x2, y2) em pixels."""
    caixas = []
    for linha in open(label).read().splitlines():
        partes = linha.split()
        if not partes:
            continue
        c, valores = int(partes[0]), [float(v) for v in partes[1:]]
        if len(valores) == 4:                      # caixa: x_centro y_centro largura altura
            xc, yc, bw, bh = valores
            x1, y1, x2, y2 = xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2
        else:                                      # polígono: usa o retângulo que o envolve
            xs, ys = valores[0::2], valores[1::2]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        caixas.append((c, int(x1 * w), int(y1 * h), int(x2 * w), int(y2 * h)))
    return caixas

def label_de(imagem):
    return imagem.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"

CORES = [(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48)]
random.seed(42)
amostras = random.sample(sorted(glob.glob(f"{ROOT}/train/images/*")), 4)

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, caminho in zip(axes.flat, amostras):
    img = cv2.cvtColor(cv2.imread(caminho), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for c, x1, y1, x2, y2 in le_caixas(label_de(caminho), w, h):
        cor = CORES[c % len(CORES)]
        cv2.rectangle(img, (x1, y1), (x2, y2), cor, 4)
        cv2.putText(img, NAMES[c], (x1, max(y1 - 10, 30)), cv2.FONT_HERSHEY_SIMPLEX, 1.4, cor, 3)
    ax.imshow(img)
    ax.set_title(os.path.basename(caminho)[:45], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{EVID}/03_amostras_treino.png", dpi=110)
plt.show()

# Conteúdo bruto do label da primeira imagem, para mostrar o formato
exemplo = f"{os.path.basename(label_de(amostras[0]))}\n{open(label_de(amostras[0])).read()}"
print(exemplo)
with open(f"{EVID}/03_exemplo_label.txt", "w") as f:
    f.write(exemplo)

## Passo 6 — Treino

Transfer learning a partir do `yolo11n.pt` (pré-treinado no COCO): 50 épocas, imagens de 640 px, batch 16.
Os pesos são salvos no Drive a cada época. Se a sessão cair, rodar esta célula de novo retoma do `last.pt`.

**Evidência:** ao terminar, tire um print do fim da saída desta célula (últimas épocas e tabela por classe).

In [ ]:
from ultralytics import YOLO

NOME = "thesis_yolo11n"
PASTA = f"{RUNS}/{NOME}"
ultimo = f"{PASTA}/weights/last.pt"

if os.path.exists(ultimo):
    try:
        YOLO(ultimo).train(resume=True)          # continua de onde a sessão anterior parou
    except AssertionError:
        print("Treino já concluído, nada a retomar:", PASTA)
else:
    model = YOLO("yolo11n.pt")
    model.train(
        data=DATA,
        epochs=50,
        imgsz=640,
        batch=16,
        project=RUNS,
        name=NOME,
        exist_ok=True,
    )

## Passo 7 — Avaliação

In [ ]:
# Avalia o melhor modelo (best.pt) no conjunto de validação
import shutil
import pandas as pd
from IPython.display import Image, display

best = YOLO(f"{PASTA}/weights/best.pt")
metrics = best.val(data=DATA, imgsz=640, project=RUNS, name="val", exist_ok=True)
print("mAP50:   ", round(metrics.box.map50, 4))
print("mAP50-95:", round(metrics.box.map, 4))

por_classe = pd.DataFrame({
    "classe": [NAMES[i] for i in metrics.box.ap_class_index],
    "precisao": metrics.box.p,
    "recall": metrics.box.r,
    "mAP50": metrics.box.ap50,
    "mAP50-95": metrics.box.ap,
}).round(3)

df = pd.read_csv(f"{PASTA}/results.csv")
df.columns = df.columns.str.strip()
ultimas = df[["epoch", "metrics/precision(B)", "metrics/recall(B)",
              "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(5)

resumo = (f"mAP50: {metrics.box.map50:.4f}\nmAP50-95: {metrics.box.map:.4f}\n\n"
          f"Métricas por classe (validação):\n{por_classe.to_string(index=False)}\n\n"
          f"Últimas épocas:\n{ultimas.to_string(index=False)}")
print("\n" + resumo)
with open(f"{EVID}/04_metricas.txt", "w") as f:
    f.write(resumo)

for arquivo in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "results.csv"]:
    shutil.copy(f"{PASTA}/{arquivo}", f"{EVID}/{arquivo}")

display(Image(f"{PASTA}/results.png", width=900))
display(Image(f"{PASTA}/confusion_matrix.png", width=650))

## Passo 8 — Uso do modelo treinado

Roda o `uso.py` do repositório (carrega o `best.pt`, detecta e salva a imagem com as caixas)
em 3 imagens do conjunto de **teste**, que o modelo nunca viu no treino.

In [ ]:
import subprocess
import sys

!wget -q -O /content/uso.py https://raw.githubusercontent.com/data-science-unicamp/aplicacoes-data-science/main/atividade3/uso.py
shutil.copy(f"{PASTA}/weights/best.pt", "/content/best.pt")

def roda_uso(imagens):
    """Executa o uso.py nas imagens, mostra o resultado e guarda as imagens anotadas nas evidências."""
    saida = subprocess.run([sys.executable, "uso.py", *imagens], cwd="/content", capture_output=True, text=True)
    print(saida.stdout)
    if saida.returncode != 0:
        print(saida.stderr)
        return
    os.makedirs(f"{EVID}/05_deteccoes", exist_ok=True)
    for imagem in imagens:
        anotada = f"/content/deteccoes/{os.path.basename(imagem)}"
        shutil.copy(anotada, f"{EVID}/05_deteccoes/")
        display(Image(anotada, width=500))

random.seed(7)
roda_uso(random.sample(sorted(glob.glob(f"{ROOT}/test/images/*")), 3))

### (Opcional) Teste com fotos suas

Fotografe uma latinha, uma garrafa PET, um papel ou uma embalagem de comida e envie em **JPG** ou **PNG**.

In [ ]:
from google.colab import files

enviadas = list(files.upload().keys())
roda_uso(enviadas)

## Passo 9 — Baixar os resultados

Junta as evidências e o `best.pt` em `resultados_atividade3.zip` e baixa para o computador.

In [ ]:
shutil.copy(f"{PASTA}/weights/best.pt", f"{EVID}/best.pt")
shutil.make_archive("/content/resultados_atividade3", "zip", EVID)
print(sorted(os.listdir(EVID)))
files.download("/content/resultados_atividade3.zip")